In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()


,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [3]:
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [4]:
df = normalize_text(df)
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [5]:
df['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})
df.head()

/var/folders/4_/5sfl5vv10mv5syqqt0j63s400000gn/T/ipykernel_84671/1089524538.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


,sentiment,content
1,0,layin n bed headache ughhhh waitin call
2,0,funeral ceremony gloomy friday
6,0,sleep im not thinking old friend want married ...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [8]:
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [9]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
import dagshub

mlflow.set_tracking_uri("https://dagshub.com/Ragrawal2004/mlops_project_emotion_detection.mlflow")
dagshub.init(repo_owner='Ragrawal2004', repo_name='mlops_project_emotion_detection', mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")



Accessing as Ragrawal2004

Initialized MLflow to track repo "Ragrawal2004/mlops_project_emotion_detection"

Repository Ragrawal2004/mlops_project_emotion_detection initialized!

2026/08/01 18:09:34 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/a690f4f12d5a4afb98cde64bd8a755de', creation_time=1785587974590, experiment_id='0', last_update_time=1785587974590, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [15]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from mlflow.models import infer_signature

# ==========================================================
# Create / Select Experiment
# ==========================================================
mlflow.set_experiment("Twitter Sentiment Detection")

# ==========================================================
# Start MLflow Run
# ==========================================================
with mlflow.start_run(run_name="LogisticRegression_BOW"):

    # ======================================================
    # Log Dataset & Preprocessing Parameters
    # ======================================================
    mlflow.log_param("dataset", "Twitter Sentiment Dataset")
    mlflow.log_param("task", "Sentiment Analysis")
    mlflow.log_param("vectorizer", "Bag of Words")
    mlflow.log_param("max_features", 1000)
    mlflow.log_param("test_size", 0.2)

    # ======================================================
    # Build Model
    # ======================================================
    model = LogisticRegression(
        C=1.0,
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    # ======================================================
    # Log Hyperparameters
    # ======================================================
    mlflow.log_param("algorithm", "Logistic Regression")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("solver", "liblinear")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)

    # ======================================================
    # Prediction
    # ======================================================
    y_pred = model.predict(X_test)

    # ======================================================
    # Metrics
    # ======================================================
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # ======================================================
    # Classification Report
    # ======================================================
    report = classification_report(y_test, y_pred)

    with open("classification_report.txt", "w") as f:
        f.write(report)

    mlflow.log_artifact("classification_report.txt")

    # ======================================================
    # Confusion Matrix
    # ======================================================
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        cmap="Blues"
    )

    plt.title("Confusion Matrix")
    plt.savefig("confusion_matrix.png")
    plt.close()

    mlflow.log_artifact("confusion_matrix.png")

    # ======================================================
    # Model Signature
    # ======================================================
    signature = infer_signature(
        X_train,
        model.predict(X_train)
    )

    # ======================================================
    # Log Model
    # ======================================================
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="twitter_sentiment_model",
        signature=signature,
        input_example=X_train[:5]
    )

    # ======================================================
    # Log Notebook
    # ======================================================
    mlflow.log_artifact("exp1.ipynb")

    # ======================================================
    # Tags
    # ======================================================
    mlflow.set_tag("author", "Rounak Agrawal")
    mlflow.set_tag("project", "Twitter Sentiment Detection")
    mlflow.set_tag("dataset", "Twitter Sentiment Dataset")
    mlflow.set_tag("task", "Sentiment Analysis")
    mlflow.set_tag("algorithm", "Logistic Regression")
    mlflow.set_tag("vectorizer", "Bag of Words")
    mlflow.set_tag("framework", "Scikit-Learn")
    mlflow.set_tag("language", "Python")
    mlflow.set_tag("mlflow_version", mlflow.__version__)

    # ======================================================
    # Print Results
    # ======================================================
    print("=" * 60)
    print("Twitter Sentiment Detection Results")
    print("=" * 60)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print("=" * 60)

2026/08/01 18:21:32 INFO mlflow.tracking.fluent: Experiment with name 'Twitter Sentiment Detection' does not exist. Creating a new experiment.


Twitter Sentiment Detection Results
Accuracy : 0.7740
Precision: 0.7650
Recall   : 0.7764
F1 Score : 0.7707
🏃 View run LogisticRegression_BOW at: https://dagshub.com/Ragrawal2004/mlops_project_emotion_detection.mlflow/#/experiments/1/runs/80ba8b0b032b4254b3f906440a70420f
🧪 View experiment at: https://dagshub.com/Ragrawal2004/mlops_project_emotion_detection.mlflow/#/experiments/1
